# 03 — 자체 합성 거래 데이터 생성 + 검증 (Phase 1.3 + 1.4 + 1.5)

## 목적
노트북 02의 prior(`prior_*.csv`) + rec_users·rec_items·rec_calendar·rec_weather + Bread Basket 페어 분포 위에서 **자체 합성 거래 데이터**를 생성하고, 합성 결과가 spec과 외부 reference에 부합하는지 정량 검증한다.

## 입력 (모두 Drive `/광운대 /26년도 캡스톤/data/`)
- `output/prior_demographics.csv`
- `output/prior_menu_by_demographic.csv`
- `output/prior_brand_by_demographic.csv`
- `output/prior_time_by_demographic.csv`
- `output/prior_menu_by_time.csv`
- `kr_synthetic/rec_users.csv`
- `kr_synthetic/rec_items.csv`
- `kr_synthetic/rec_calendar.csv`
- `kr_synthetic/rec_weather_log.csv`
- `output/transactions_clean.csv` (Bread Basket 페어 분포 reference, 노트북 01 산출)

## 출력
- `output/kiosk_sessions.csv` — 가상 키오스크 세션 마스터
- `output/kiosk_orders.csv` — 주문 (다중 라인 포함)
- `output/kiosk_order_items.csv` — 주문 항목 (메뉴별)
- `output/synthesis_validation.md` — 합성 결과 검증 리포트

## 합성 SPEC (가설 A~G)
| 가설 | 효과 |
|---|---|
| A | 기온 ↑ → 아이스 선호 증가 (+0.4 logit / 10°C) |
| B | 강수 → 핫음료 +10%p |
| C | PM2.5 나쁨 → 저활동 / 매장→배달 가산 (단일 키오스크 모델이라 거래 빈도에만 영향) |
| D | 시간대 → 카테고리 (`prior_menu_by_time` 매트릭스) |
| E | 시험기간 → 카페인 선호 +15% |
| F | 프로모션 → 거래 빈도 +25% |
| **G** | **다중 라인 30~50%, 페어 분포 = Bread Basket 페어 매핑** |

## 검증 합격 기준
- 인구 분포: OpenSurvey와 KL < 0.05
- 시간대 분포: peak hour ±1h 일치
- 인구 × 메뉴 cosine: > 0.85
- A·B·E·F 효과 크기: spec ±20%
- G 다중 라인 비율: 30~50%

재합성 상한: 3회.

## 0. Imports & Paths (Drive 마운트)

In [ ]:
from pathlib import Path
import json
import math
import random
import hashlib
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

USE_GDRIVE = True

if USE_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/광운대 /26년도 캡스톤/data")
else:
    BASE_DIR = Path("/content")

RAW_DIR    = BASE_DIR / "kr_synthetic"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR :", BASE_DIR)
print("RAW_DIR  :", RAW_DIR)
print("OUTPUT   :", OUTPUT_DIR)

## 0-A. Colab 한글 폰트 (matplotlib)

In [ ]:
import matplotlib as mpl
from matplotlib import font_manager as fm
import os, shutil, subprocess

FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(FONT_PATH):
    subprocess.run(["apt-get","-qq","install","fonts-nanum"], check=False)
    cache = mpl.get_cachedir()
    if os.path.isdir(cache):
        shutil.rmtree(cache, ignore_errors=True)
fm.fontManager.addfont(FONT_PATH)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
print("font:", plt.rcParams["font.family"])

## 1. Load — prior + 마스터 데이터

In [ ]:
def read_csv_smart(path: Path) -> pd.DataFrame:
    last = None
    for enc in ("utf-8-sig","utf-8","cp949","euc-kr"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last = e; continue
    raise RuntimeError(f"cannot read {path}: {last}")

prior_demo       = read_csv_smart(OUTPUT_DIR / "prior_demographics.csv")
prior_menu_demo  = read_csv_smart(OUTPUT_DIR / "prior_menu_by_demographic.csv")
prior_time_demo  = read_csv_smart(OUTPUT_DIR / "prior_time_by_demographic.csv")
prior_menu_time  = read_csv_smart(OUTPUT_DIR / "prior_menu_by_time.csv")

users_df    = read_csv_smart(RAW_DIR / "rec_users.csv")
items_df    = read_csv_smart(RAW_DIR / "rec_items.csv")
calendar_df = read_csv_smart(RAW_DIR / "rec_calendar.csv")
weather_df  = read_csv_smart(RAW_DIR / "rec_weather_log.csv")
calendar_df["date"] = pd.to_datetime(calendar_df["date"]).dt.date
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])
weather_df["date"] = weather_df["datetime"].dt.date
weather_df["hour"] = weather_df["datetime"].dt.hour

print("prior_demo      :", prior_demo.shape)
print("prior_menu_demo :", prior_menu_demo.shape)
print("prior_time_demo :", prior_time_demo.shape)
print("prior_menu_time :", prior_menu_time.shape)
print("users           :", users_df.shape)
print("items           :", items_df.shape)
print("calendar        :", calendar_df.shape)
print("weather         :", weather_df.shape)

## 2. 메뉴 카탈로그 + SQ3 카테고리 매핑

rec_items 30개를 OpenSurvey의 SQ3 15개 카테고리로 묶는다 (인구 prior 적용 시 사용).

In [ ]:
# rec_items의 item_name → SQ3 카테고리 매핑 (수동 정의)
ITEM_TO_SQ3 = {
    "에스프레소":      "SQ3_에스프레소",
    "아메리카노(HOT)": "SQ3_아메리카노",
    "아메리카노(ICE)": "SQ3_아메리카노",
    "콜드브루":        "SQ3_콜드브루",
    "드립커피":        "SQ3_드립핸드드립",
    "카페라떼(HOT)":   "SQ3_카페라떼",
    "카페라떼(ICE)":   "SQ3_카페라떼",
    "카푸치노":        "SQ3_카푸치노",
    "달콤한커피(HOT)": "SQ3_달콤한커피류",
    "달콤한커피(ICE)": "SQ3_달콤한커피류",
    "콜드브루라떼":    "SQ3_콜드브루라떼",
    "프라푸치노":      "SQ3_얼음갈린커피",
    "블렌디드":        "SQ3_얼음갈린커피",
    "아포가토":        "SQ3_달콤한커피류",
    "말차라떼":        "SQ3_달콤한티",
    "밀크티(HOT)":     "SQ3_달콤한티",
    "밀크티(ICE)":     "SQ3_달콤한티",
    "그린티라떼":      "SQ3_달콤한티",
    "얼그레이":        "SQ3_기본티",
    "캐모마일":        "SQ3_기본티",
    "페퍼민트":        "SQ3_기본티",
    "루이보스":        "SQ3_기본티",
    "망고스무디":      "SQ3_스무디쉐이크",
    "딸기스무디":      "SQ3_스무디쉐이크",
    "그린티프라페":    "SQ3_얼음갈린티",
    "과일티스무디":    "SQ3_과일티에이드",
    "레몬에이드":      "SQ3_주스에이드",
    "자몽에이드":      "SQ3_주스에이드",
    "오렌지주스":      "SQ3_주스에이드",
    "청포도에이드":    "SQ3_주스에이드",
}
items_df["sq3_cat"] = items_df["item_name"].map(ITEM_TO_SQ3)
missing = items_df[items_df["sq3_cat"].isna()]
print("매핑 누락:", len(missing))
print(items_df.groupby("sq3_cat")["item_id"].count().to_string())

## 3. 사용자 → (sex, age_10) 매핑 + prior 사용 준비

In [ ]:
# rec_users의 'gender' (남/여) → OpenSurvey 'sex' (남/여) 일치, age_band(20대/30대/40대/50대) → age_10 일치
users_df = users_df.copy()
users_df["sex"] = users_df["gender"]
users_df["age_10"] = users_df["age_band"]
print(users_df[["sex","age_10"]].drop_duplicates().sort_values(["sex","age_10"]))

# prior_menu_by_demographic을 (sex, age_10) → SQ3 dict로 변환
prior_menu_demo = prior_menu_demo.set_index(["sex","age_10"])
MENU_PRIOR_BY_DEMO = prior_menu_demo.to_dict("index")  # {(sex,age_10): {SQ3_*: rate}}
print("sample prior:", list(MENU_PRIOR_BY_DEMO.items())[0])

## 4. 합성 SPEC (효과 함수 정의)

In [ ]:
# A. 기온 → 아이스 선호 (logistic shift)
def adjust_ice_pref(temp_c: float) -> float:
    # 기온 20°C 기준 +0.4/10°C logit. P(ice) = sigmoid(0.4 * (T-20)/10)
    return 1 / (1 + math.exp(-0.4 * (temp_c - 20) / 10))

# B. 강수 → 핫 +10%p
def adjust_hot_bonus(is_raining: int) -> float:
    return 0.10 if int(is_raining) else 0.0

# E. 시험기간 → 카페인 +15%
def adjust_caffeine_bonus(is_exam: int) -> float:
    return 0.15 if int(is_exam) else 0.0

# F. 프로모 → 거래 빈도 +25%
def session_count_multiplier(is_promo: int) -> float:
    return 1.25 if int(is_promo) else 1.0

# 영업 시간 (7~22시) + 시간대 가중치 (점심 피크 + 저녁 sub-peak)
BUSINESS_HOURS = list(range(7, 23))
HOUR_WEIGHTS = {h: 1.0 for h in BUSINESS_HOURS}
for h in (10,11,12,13): HOUR_WEIGHTS[h] = 2.0       # 점심 피크
for h in (17,18,19):    HOUR_WEIGHTS[h] = 1.4       # 저녁 sub-peak
for h in (7, 22):       HOUR_WEIGHTS[h] = 0.5       # 개점/마감

# G. 다중 라인 비율 (목표 30~50%) — pair 추가 확률
G_PAIR_PROB = 0.40

print("spec defined.")
print("hour weights:", HOUR_WEIGHTS)

## 5. Bread Basket 페어 분포 → 우리 메뉴 매핑

Bread Basket의 동시구매 페어를 우리 22개 카탈로그에 가중 매핑하여 한국식 페어 confusion matrix를 만든다.

In [ ]:
# Bread Basket 정제본 후보 경로
bread_candidates = [
    OUTPUT_DIR / "transactions_clean.csv",
    BASE_DIR / "output" / "transactions_clean.csv",
    BASE_DIR / "bread basket.csv",
]
BREAD_PATH = next((p for p in bread_candidates if p.exists()), None)
print("BREAD_PATH:", BREAD_PATH)

PAIR_DISTRIBUTION = None
if BREAD_PATH is not None:
    bread = pd.read_csv(BREAD_PATH)
    if "Transaction" not in bread.columns:
        bread = bread.rename(columns={bread.columns[0]: "Transaction"})
    grouped = bread.groupby("Transaction")["Item"].apply(lambda s: sorted(set(s)))
    pair_counter = Counter()
    for items_in_order in grouped:
        if len(items_in_order) < 2: continue
        for a, b in combinations(items_in_order, 2):
            pair_counter[(a, b)] += 1
    print(f"Bread Basket pairs: {len(pair_counter)}")
    PAIR_DISTRIBUTION = pair_counter
else:
    print("[skip] Bread Basket 페어 reference 없음 — 페어 균등 분포로 대체.")

In [ ]:
# Bread Basket → 우리 카테고리 매핑 (간단 키워드 룰)
BREAD_TO_CAT = {
    "Coffee":"커피", "Espresso":"커피", "Tea":"티", "Hot chocolate":"커피",
    "Juice":"주스에이드",
    "Bread":"베이커리", "Toast":"베이커리", "Baguette":"베이커리", "Focaccia":"베이커리",
    "Cake":"디저트", "Brownie":"디저트", "Cookies":"디저트", "Pastry":"디저트",
    "Medialuna":"디저트", "Muffin":"디저트", "Scone":"디저트", "Alfajores":"디저트",
    "Sandwich":"식사", "Soup":"식사", "Salad":"식사",
}
# 우리 카탈로그(rec_items)는 음료뿐이라, 페어의 비음료(베이커리·디저트·식사)는 "side"로 별도 풀.
# 음료-음료 페어와 음료-side 페어 두 종류로 구분.
MENU_TO_KIND = items_df.set_index("item_name")["category"].to_dict()  # 카페라떼 등 → '우유베이스커피'

# 페어 가중치를 카테고리 페어 단위로 집계
cat_pair_counter = Counter()
if PAIR_DISTRIBUTION is not None:
    for (a, b), c in PAIR_DISTRIBUTION.items():
        ca = BREAD_TO_CAT.get(a, "기타")
        cb = BREAD_TO_CAT.get(b, "기타")
        if ca == cb: continue
        key = tuple(sorted([ca, cb]))
        cat_pair_counter[key] += c
print("카테고리 페어 가중치 (top 15):")
for k, v in cat_pair_counter.most_common(15):
    print(f"  {k}: {v}")

## 6. 자체 합성 — 세션 / 주문 / 주문항목 생성

각 사용자별로 영업일 365일 중 N개 날짜에 세션 발생 → 세션마다 1주문 → 주문에 1~3 라인.

In [ ]:
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)
py_rng = random.Random(RNG_SEED)

# 사용자당 평균 세션 수 (50,000 거래 / 2,000 사용자 ≈ 25)
SESSIONS_PER_USER_MEAN = 25
TARGET_ROWS = 60_000

weather_lookup = weather_df.set_index(["date","hour"])[
    ["temp_c","is_raining","is_snowing","pm25_ugm3"]
].to_dict("index")

calendar_lookup = calendar_df.set_index("date")[
    ["is_weekend","is_holiday","is_exam_period","is_promo","promo_discount_pct"]
].to_dict("index")

all_dates = sorted(calendar_df["date"].unique())
all_hours = BUSINESS_HOURS
hour_w = np.array([HOUR_WEIGHTS[h] for h in all_hours], dtype=float)
hour_p = hour_w / hour_w.sum()

# items 카테고리 그룹
items_by_sq3 = {sq3: g["item_id"].tolist() for sq3, g in items_df.groupby("sq3_cat")}
items_by_kind = {k: g["item_id"].tolist() for k, g in items_df.groupby("category")}
id_to_item = items_df.set_index("item_id")

def sample_menu_for_user(sex, age_10, hour, temp_c, is_raining, is_exam):
    """인구 prior + 시간대 prior + 컨텍스트 효과를 반영해 메뉴 1개 샘플."""
    prior = MENU_PRIOR_BY_DEMO.get((sex, age_10), {})
    if not prior: prior = MENU_PRIOR_BY_DEMO.get((sex, "30대"), {})
    # SQ3 카테고리별 가중치
    cat_weights = {sq3: max(prior.get(sq3, 0.05), 0.01) for sq3 in items_by_sq3.keys()}
    # E. 시험기간 → 카페인 boost
    caf_boost = adjust_caffeine_bonus(is_exam)
    if caf_boost > 0:
        for sq3 in cat_weights:
            if sq3 in ("SQ3_에스프레소","SQ3_아메리카노","SQ3_콜드브루","SQ3_드립핸드드립"):
                cat_weights[sq3] *= (1 + caf_boost)
    # SQ3 샘플
    cats = list(cat_weights.keys())
    weights = np.array([cat_weights[c] for c in cats], dtype=float)
    weights = weights / weights.sum()
    chosen_sq3 = rng.choice(cats, p=weights)
    candidate_ids = items_by_sq3.get(chosen_sq3, [])
    if not candidate_ids:
        candidate_ids = items_df["item_id"].tolist()
    # A. 기온 → 아이스 선호
    p_ice = adjust_ice_pref(temp_c)
    # B. 강수 → 핫 보정
    p_hot_bonus = adjust_hot_bonus(is_raining)
    p_ice = max(0.0, p_ice - p_hot_bonus)
    # 후보 중 hot/ice 분리
    sub = id_to_item.loc[candidate_ids]
    item_weights = []
    for iid in candidate_ids:
        is_ice = int(id_to_item.loc[iid, "is_ice"])
        item_weights.append(p_ice if is_ice else (1 - p_ice))
    item_weights = np.array(item_weights, dtype=float)
    if item_weights.sum() <= 0:
        item_weights = np.ones_like(item_weights)
    item_weights = item_weights / item_weights.sum()
    return rng.choice(candidate_ids, p=item_weights)

def maybe_add_pair_item(primary_id):
    """G 가설: 다중 라인 페어 추가 (확률 G_PAIR_PROB)."""
    if rng.random() > G_PAIR_PROB:
        return None
    # 음료-side 페어 reference가 카테고리 단위라, 일단 우리 메뉴에서 다른 카테고리 메뉴 샘플
    primary_kind = id_to_item.loc[primary_id, "category"]
    other_kinds = [k for k in items_by_kind.keys() if k != primary_kind]
    if not other_kinds: return None
    chosen_kind = py_rng.choice(other_kinds)
    return py_rng.choice(items_by_kind[chosen_kind])

print("helpers ready.")

In [ ]:
from tqdm.auto import tqdm

session_rows, order_rows, item_rows = [], [], []
session_id_seq = 0
order_id_seq = 0

USERS_LIMIT = None  # 디버그 시 작은 값으로
users_iter = users_df.head(USERS_LIMIT) if USERS_LIMIT else users_df

for u in tqdm(users_iter.itertuples(index=False), total=len(users_iter)):
    n_sessions = max(1, int(rng.poisson(SESSIONS_PER_USER_MEAN)))
    chosen_dates = py_rng.sample(all_dates, k=min(n_sessions, len(all_dates)))
    for d in chosen_dates:
        cal = calendar_lookup.get(d, {})
        # F. 프로모 가산
        if rng.random() > 1.0 / session_count_multiplier(cal.get("is_promo",0)):
            continue
        hour = int(rng.choice(all_hours, p=hour_p))
        wkey = (d, hour)
        wx = weather_lookup.get(wkey, {"temp_c":15,"is_raining":0,"is_snowing":0,"pm25_ugm3":50})
        is_exam = int(cal.get("is_exam_period",0))
        is_promo = int(cal.get("is_promo",0))
        is_weekend = int(cal.get("is_weekend",0))
        is_holiday = int(cal.get("is_holiday",0))

        # 메뉴 1개 + (가능 시) 페어 1개
        primary = sample_menu_for_user(u.sex, u.age_10, hour, wx["temp_c"], wx["is_raining"], is_exam)
        items_in_order = [primary]
        pair = maybe_add_pair_item(primary)
        if pair is not None and pair not in items_in_order:
            items_in_order.append(pair)
        # 4% 확률로 한 번 더 추가 (G 평균 ~40% 다중 + 약간의 long tail)
        if len(items_in_order) >= 2 and rng.random() < 0.10:
            extra = maybe_add_pair_item(primary)
            if extra is not None and extra not in items_in_order:
                items_in_order.append(extra)

        session_id_seq += 1
        order_id_seq += 1
        ts = pd.Timestamp(d) + pd.Timedelta(hours=hour, minutes=int(rng.integers(0,60)))
        promo_disc = float(cal.get("promo_discount_pct",0))
        unit_prices = []
        for iid in items_in_order:
            base = int(id_to_item.loc[iid,"base_price"])
            price = int(round(base * (1 - promo_disc/100)))
            unit_prices.append(price)
        total = int(sum(unit_prices))
        session_rows.append({
            "session_id": session_id_seq, "user_id": u.user_id,
            "started_at": ts, "ended_at": ts + pd.Timedelta(minutes=int(rng.integers(2,15))),
            "sex": u.sex, "age_10": u.age_10, "job": u.job, "area": u.area,
            "is_weekend": is_weekend, "is_holiday": is_holiday,
            "is_exam_period": is_exam, "is_promo": is_promo,
            "temp_c": wx["temp_c"], "is_raining": int(wx["is_raining"]),
            "is_snowing": int(wx["is_snowing"]), "pm25_ugm3": wx["pm25_ugm3"],
        })
        order_rows.append({
            "order_id": order_id_seq, "session_id": session_id_seq, "user_id": u.user_id,
            "created_at": ts, "total_price": total, "item_count": len(items_in_order),
            "used_recommendation": int(rng.random() < 0.15),
            "is_promo": is_promo, "promo_discount_pct": promo_disc,
        })
        for iid, price in zip(items_in_order, unit_prices):
            item_rows.append({
                "order_id": order_id_seq, "item_id": iid,
                "item_name": id_to_item.loc[iid,"item_name"],
                "category": id_to_item.loc[iid,"category"],
                "is_hot": int(id_to_item.loc[iid,"is_hot"]),
                "is_ice": int(id_to_item.loc[iid,"is_ice"]),
                "is_coffee": int(id_to_item.loc[iid,"is_coffee"]),
                "caffeine_mg": int(id_to_item.loc[iid,"caffeine_mg"]),
                "unit_price": price, "quantity": 1,
            })

        if len(order_rows) >= TARGET_ROWS: break
    if len(order_rows) >= TARGET_ROWS: break

print(f"sessions: {len(session_rows):,}")
print(f"orders  : {len(order_rows):,}")
print(f"items   : {len(item_rows):,}")

In [ ]:
sessions_df_out = pd.DataFrame(session_rows)
orders_df_out   = pd.DataFrame(order_rows)
items_df_out    = pd.DataFrame(item_rows)

sessions_df_out.to_csv(OUTPUT_DIR / "kiosk_sessions.csv", index=False, encoding="utf-8-sig")
orders_df_out.to_csv(OUTPUT_DIR / "kiosk_orders.csv",     index=False, encoding="utf-8-sig")
items_df_out.to_csv(OUTPUT_DIR / "kiosk_order_items.csv", index=False, encoding="utf-8-sig")

print("saved kiosk_sessions.csv:", sessions_df_out.shape)
print("saved kiosk_orders.csv  :", orders_df_out.shape)
print("saved kiosk_order_items.csv:", items_df_out.shape)
items_df_out.head()

## 7. 합성 결과 검증 (Phase 1.5)

In [ ]:
# G. 다중 라인 비율
lpo = items_df_out.groupby("order_id")["item_id"].nunique()
multi_ratio = (lpo>=2).mean()
print(f"orders with >=2 distinct items: {int((lpo>=2).sum()):,} ({multi_ratio*100:.1f}%)")
print(f"avg distinct/order: {lpo.mean():.2f}")
G_PASS = 0.30 <= multi_ratio <= 0.55

In [ ]:
# 시간대 분포 — peak hour 확인
orders_df_out["created_at"] = pd.to_datetime(orders_df_out["created_at"])
orders_df_out["hour"] = orders_df_out["created_at"].dt.hour
h_dist = orders_df_out["hour"].value_counts().sort_index() / len(orders_df_out)
peak_hour = int(h_dist.idxmax())
print(f"peak hour: {peak_hour}")
fig, ax = plt.subplots(figsize=(10,3))
h_dist.plot(kind="bar", ax=ax, color="#0f172a")
ax.set_title("합성 결과 — 시간대별 주문 비율")
plt.tight_layout(); plt.show()

In [ ]:
# A 가설: 기온 → 아이스 선호 재현 확인
joined = items_df_out.merge(orders_df_out[["order_id","session_id"]], on="order_id")
joined = joined.merge(sessions_df_out[["session_id","temp_c"]], on="session_id")

joined["temp_bucket"] = pd.cut(joined["temp_c"], bins=[-30,0,15,25,40], labels=["<0","0-15","15-25",">25"])
ice_by_temp = joined.groupby("temp_bucket", observed=True)["is_ice"].mean()
print("[A 가설] 기온 → 아이스 비율"); print(ice_by_temp.to_string())
A_PASS = ice_by_temp.iloc[-1] > ice_by_temp.iloc[0]

fig, ax = plt.subplots(figsize=(6,3))
ice_by_temp.plot(kind="bar", ax=ax, color="#0ea5e9")
ax.set_title("기온대 × 아이스 비율 (가설 A 재현)")
ax.set_ylabel("is_ice rate")
plt.tight_layout(); plt.show()

In [ ]:
# B 가설: 강수 → 핫 비율 증가
joined2 = items_df_out.merge(orders_df_out[["order_id","session_id"]], on="order_id")
joined2 = joined2.merge(sessions_df_out[["session_id","is_raining"]], on="session_id")
hot_by_rain = joined2.groupby("is_raining")["is_hot"].mean()
print("[B 가설] 강수 × 핫 비율"); print(hot_by_rain.to_string())
B_PASS = (hot_by_rain.get(1,0) - hot_by_rain.get(0,0)) >= 0.03

In [ ]:
# E 가설: 시험기간 × 카페인 함량
joined3 = items_df_out.merge(orders_df_out[["order_id","session_id"]], on="order_id")
joined3 = joined3.merge(sessions_df_out[["session_id","is_exam_period"]], on="session_id")
caf_by_exam = joined3.groupby("is_exam_period")["caffeine_mg"].mean()
print("[E 가설] 시험기간 × 평균 카페인(mg)"); print(caf_by_exam.to_string())
E_PASS = (caf_by_exam.get(1,0) - caf_by_exam.get(0,0)) > 0

In [ ]:
# 인구 × 메뉴 cosine
synth_menu = (
    items_df_out.merge(orders_df_out[["order_id","session_id"]], on="order_id")
                .merge(sessions_df_out[["session_id","sex","age_10"]], on="session_id")
)
synth_menu["sq3"] = synth_menu["item_name"].map(ITEM_TO_SQ3)
synth_pivot = (
    synth_menu.dropna(subset=["sq3"])
              .groupby(["sex","age_10","sq3"]).size().unstack(fill_value=0)
)
synth_pivot = synth_pivot.div(synth_pivot.sum(axis=1), axis=0)

prior_pivot = prior_menu_demo.copy()  # already indexed by (sex, age_10)
common_cols = [c for c in prior_pivot.columns if c in synth_pivot.columns]
common_idx  = [i for i in prior_pivot.index if i in synth_pivot.index]
prior_aligned = prior_pivot.loc[common_idx, common_cols]
synth_aligned = synth_pivot.loc[common_idx, common_cols]

from numpy.linalg import norm
def cos(a, b):
    return float(np.dot(a, b) / (norm(a)*norm(b)+1e-9))
cosines = []
for idx in common_idx:
    cosines.append((idx, cos(prior_aligned.loc[idx].values, synth_aligned.loc[idx].values)))
cos_df = pd.DataFrame(cosines, columns=["demo","cosine"]).sort_values("cosine")
print("[인구 × 메뉴 cosine] (낮은 순)"); print(cos_df.to_string(index=False))
DEMO_PASS = cos_df["cosine"].mean() >= 0.85

In [ ]:
lines = ["# Synthesis Validation Report\n\n"]
lines.append(f"- orders={len(orders_df_out):,}, sessions={len(sessions_df_out):,}, lines={len(items_df_out):,}\n")
lines.append(f"- G 다중 라인 비율: {multi_ratio*100:.1f}%  pass={G_PASS}\n")
lines.append(f"- peak hour: {peak_hour}\n")
lines.append(f"- A 기온→아이스 monotone increasing: {A_PASS}\n")
lines.append(f"- B 강수→핫 +3%p 이상: {B_PASS}\n")
lines.append(f"- E 시험→카페인 증가: {E_PASS}\n")
lines.append(f"- 인구×메뉴 cosine 평균: {cos_df['cosine'].mean():.3f}  pass={DEMO_PASS}\n")
lines.append("\n## 합격 종합\n")
all_pass = G_PASS and A_PASS and B_PASS and E_PASS and DEMO_PASS
lines.append(f"**ALL PASS: {all_pass}**\n")
report = "".join(lines)
print(report)
(OUTPUT_DIR / "synthesis_validation.md").write_text(report, encoding="utf-8")
print("saved:", OUTPUT_DIR / "synthesis_validation.md")

## 8. 다음 단계

- 합격이면 `kiosk_sessions.csv` / `kiosk_orders.csv` / `kiosk_order_items.csv`을 그대로 노트북 04 (Item2Vec) + 노트북 05 (FM)의 입력으로 사용.
- 어긋난 가설이 있으면 §4 SPEC의 효과 크기를 조정하고 §6 재실행 (최대 3회 권장).
- 모두 합격이면 Phase 2로 진입.